In [ ]:
from copy import deepcopy
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


TASKS = ["Baseline", "Logic", "Nback", "Stroop", "Sudoku"]
STRESS_TASKS = {"Logic", "Nback", "Stroop", "Sudoku"}
ACC_SAMPLING_RATE_HZ = 32


@dataclass
class HyperParameters:
    random_seed: int = 42
    val_ratio: float = 0.1
    min_val_subjects: int = 4
    max_val_subjects: int = 5

    window_size: int = 1920
    stride: int = 320
    batch_size: int = 64
    epochs: int = 50
    learning_rate: float = 1e-3
    dropout_rate: float = 0.3
    weight_decay: float = 1e-4
    patience: int = 8

    transformer_heads: int = 8
    transformer_layers: int = 2
    transformer_ff_dim: int = 256
    transformer_dropout: float = 0.1

    # subject-wise zscore | baseline zscore | none
    normalization_mode: str = "subject_zscore"

    # focal | weighted_bce
    loss_name: str = "focal"
    focal_gamma: float = 2.0
    focal_alpha: Optional[float] = None
    save_model: bool = True

    # 개발/디버깅용: None이면 전체 LOSO fold 수행
    max_folds: Optional[int] = None


def find_dataset_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "Dataset" / "CATSA"
        if candidate.exists():
            return candidate

    fallback = Path("/home/binghin2/Myproject/Dataset/CATSA")
    if fallback.exists():
        return fallback

    raise FileNotFoundError("Could not find Dataset/CATSA. Please set dataset_root manually.")


def discover_complete_subjects(dataset_root: Path) -> List[str]:
    subjects: List[str] = []
    for subject_dir in sorted(dataset_root.glob("Sub*"), key=lambda p: int(p.name[3:])):
        if not subject_dir.is_dir():
            continue
        has_all = all((subject_dir / task / "ACC.csv").exists() for task in TASKS)
        if has_all:
            subjects.append(subject_dir.name)
    return subjects


def read_acc_csv(file_path: Path) -> np.ndarray:
    df = pd.read_csv(file_path)
    expected_cols = ["ACC_X", "ACC_Y", "ACC_Z"]

    if all(col in df.columns for col in expected_cols):
        values = df[expected_cols].to_numpy(dtype=np.float32)
    else:
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(numeric_cols) < 3:
            raise ValueError(f"ACC must have at least 3 numeric columns in {file_path}")
        values = df[numeric_cols[:3]].to_numpy(dtype=np.float32)

    mask = ~np.any(np.isnan(values), axis=1)
    values = values[mask]
    if len(values) == 0:
        raise ValueError(f"No valid ACC values in {file_path}")
    return values


def sliding_windows(signal: np.ndarray, window_size: int, stride: int) -> np.ndarray:
    if signal.ndim != 2:
        raise ValueError(f"Expected 2D ACC signal [T, C], got shape={signal.shape}")

    if len(signal) < window_size:
        return np.empty((0, signal.shape[1], window_size), dtype=np.float32)

    windows = [signal[i:i + window_size].T for i in range(0, len(signal) - window_size + 1, stride)]
    return np.asarray(windows, dtype=np.float32)


def compute_subject_stats(sub_path: Path, use_baseline_as_non_stress: bool, use_stress_tasks_only: bool) -> Tuple[np.ndarray, np.ndarray]:
    if use_stress_tasks_only:
        candidate_tasks = [t for t in TASKS if t in STRESS_TASKS]
    else:
        candidate_tasks = TASKS.copy()

    values_list: List[np.ndarray] = []
    for task in candidate_tasks:
        if task == "Baseline" and not use_baseline_as_non_stress:
            continue
        values_list.append(read_acc_csv(sub_path / task / "ACC.csv"))

    all_values = np.concatenate(values_list, axis=0)
    sub_mean = np.mean(all_values, axis=0).astype(np.float32)
    sub_std = (np.std(all_values, axis=0) + 1e-8).astype(np.float32)
    return sub_mean, sub_std


def build_split_arrays(
    dataset_root: Path,
    subject_list: List[str],
    window_size: int,
    stride: int,
    normalization_mode: str = "subject_zscore",
    use_baseline_as_non_stress: bool = True,
    use_stress_tasks_only: bool = False,
) -> Tuple[np.ndarray, np.ndarray]:
    x_chunks: List[np.ndarray] = []
    y_chunks: List[np.ndarray] = []

    for sub in subject_list:
        sub_path = dataset_root / sub

        if use_stress_tasks_only:
            candidate_tasks = [t for t in TASKS if t in STRESS_TASKS]
        else:
            candidate_tasks = TASKS.copy()

        baseline = read_acc_csv(sub_path / "Baseline" / "ACC.csv")
        baseline_mean = np.mean(baseline, axis=0).astype(np.float32)
        baseline_std = (np.std(baseline, axis=0) + 1e-8).astype(np.float32)

        sub_mean, sub_std = compute_subject_stats(
            sub_path,
            use_baseline_as_non_stress=use_baseline_as_non_stress,
            use_stress_tasks_only=use_stress_tasks_only,
        )

        for task in candidate_tasks:
            if task == "Baseline" and not use_baseline_as_non_stress:
                continue

            raw = read_acc_csv(sub_path / task / "ACC.csv")
            if normalization_mode == "subject_zscore":
                norm = (raw - sub_mean) / sub_std
            elif normalization_mode == "baseline_zscore":
                norm = (raw - baseline_mean) / baseline_std
            elif normalization_mode == "none":
                norm = raw
            else:
                raise ValueError(f"Unknown normalization_mode: {normalization_mode}")

            windows = sliding_windows(norm, window_size=window_size, stride=stride)
            if len(windows) == 0:
                continue

            label = 1 if task in STRESS_TASKS else 0
            labels = np.full((len(windows),), label, dtype=np.int64)
            x_chunks.append(windows)
            y_chunks.append(labels)

    if not x_chunks:
        raise ValueError("No samples generated. Check data files or window settings.")

    x = np.concatenate(x_chunks, axis=0)
    y = np.concatenate(y_chunks, axis=0)
    return x.astype(np.float32), y.astype(np.int64)


def build_loso_folds(subjects: List[str], hp: HyperParameters) -> List[Dict[str, List[str]]]:
    folds: List[Dict[str, List[str]]] = []
    for fold_idx, test_subject in enumerate(subjects):
        remaining = [s for s in subjects if s != test_subject]

        val_count = int(round(len(remaining) * hp.val_ratio))
        val_count = max(hp.min_val_subjects, val_count)
        val_count = min(hp.max_val_subjects, val_count)
        val_count = min(val_count, len(remaining) - 1)

        rng = np.random.default_rng(hp.random_seed + fold_idx)
        shuffled = remaining.copy()
        rng.shuffle(shuffled)

        val_subjects = sorted(shuffled[:val_count], key=lambda x: int(x[3:]))
        train_subjects = sorted(shuffled[val_count:], key=lambda x: int(x[3:]))

        folds.append({
            "train": train_subjects,
            "val": val_subjects,
            "test": [test_subject],
        })

    return folds


class Acc1DCNNTransformer(nn.Module):
    def __init__(self, hp: HyperParameters):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=hp.transformer_heads,
            dim_feedforward=hp.transformer_ff_dim,
            dropout=hp.transformer_dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=hp.transformer_layers,
        )

        self.max_len = max(16, hp.window_size)
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.max_len, 128))

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(hp.dropout_rate),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cnn(x)
        x = x.transpose(1, 2)  # [B, L, C]

        seq_len = x.size(1)
        if seq_len > self.pos_embedding.size(1):
            pos = nn.functional.interpolate(
                self.pos_embedding.transpose(1, 2),
                size=seq_len,
                mode="linear",
                align_corners=False,
            ).transpose(1, 2)
        else:
            pos = self.pos_embedding[:, :seq_len, :]

        x = x + pos
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x).squeeze(1)


class BinaryFocalLossWithLogits(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha: Optional[float] = None, pos_weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        if pos_weight is not None:
            self.register_buffer("pos_weight", pos_weight)
        else:
            self.pos_weight = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
            pos_weight=self.pos_weight,
        )
        probs = torch.sigmoid(logits)
        pt = probs * targets + (1.0 - probs) * (1.0 - targets)
        focal_factor = (1.0 - pt).pow(self.gamma)

        if self.alpha is not None:
            alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
            loss = alpha_t * focal_factor * bce
        else:
            loss = focal_factor * bce

        return loss.mean()


def compute_pos_weight(y: np.ndarray) -> torch.Tensor:
    pos = float(np.sum(y == 1))
    neg = float(np.sum(y == 0))
    if pos == 0:
        return torch.tensor([1.0], dtype=torch.float32)
    return torch.tensor([neg / pos], dtype=torch.float32)


def make_criterion(y_train: np.ndarray, hp: HyperParameters, device: torch.device) -> Tuple[nn.Module, Dict[str, object]]:
    pos_weight = compute_pos_weight(y_train).to(device)

    if hp.loss_name == "weighted_bce":
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight), {
            "loss_name": "weighted_bce",
            "pos_weight": float(pos_weight.item()),
            "focal_gamma": 0.0,
            "focal_alpha": 0.0,
        }

    if hp.loss_name == "focal":
        pos = float(np.sum(y_train == 1))
        neg = float(np.sum(y_train == 0))
        alpha = hp.focal_alpha
        if alpha is None:
            alpha = neg / max(pos + neg, 1.0)

        criterion = BinaryFocalLossWithLogits(
            gamma=hp.focal_gamma,
            alpha=float(alpha),
            pos_weight=None,
        )
        return criterion, {
            "loss_name": "focal",
            "pos_weight": 1.0,
            "focal_gamma": float(hp.focal_gamma),
            "focal_alpha": float(alpha),
            "weighting_note": "Focal uses alpha only to avoid duplicate class weighting",
        }

    raise ValueError(f"Unknown loss_name: {hp.loss_name}")


def make_loader(x: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y).float())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total = 0
    tp = tn = fp = fn = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += float(loss.item()) * xb.size(0)
        total += xb.size(0)

        preds = (torch.sigmoid(logits) >= 0.5).long()
        y_int = yb.long()

        tp += int(((preds == 1) & (y_int == 1)).sum().item())
        tn += int(((preds == 0) & (y_int == 0)).sum().item())
        fp += int(((preds == 1) & (y_int == 0)).sum().item())
        fn += int(((preds == 0) & (y_int == 1)).sum().item())

    acc = (tp + tn) / max(total, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tn": float(tn),
        "fp": float(fp),
        "fn": float(fn),
        "tp": float(tp),
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    clip_grad: float = 1.0,
) -> float:
    model.train()
    total_loss = 0.0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
        optimizer.step()

        total_loss += float(loss.item()) * xb.size(0)
        total += xb.size(0)

    return total_loss / max(total, 1)


def run_loso_training(
    model_name: str = "acc_1dcnn_transformer_loso_torch",
    hp: HyperParameters = HyperParameters(),
) -> Dict[str, object]:
    np.random.seed(hp.random_seed)
    torch.manual_seed(hp.random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(hp.random_seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset_root = find_dataset_root()

    save_dir = Path("/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ACC/Save_model")
    save_dir.mkdir(parents=True, exist_ok=True)

    subjects = discover_complete_subjects(dataset_root)
    folds = build_loso_folds(subjects, hp)
    if hp.max_folds is not None:
        folds = folds[: hp.max_folds]

    fold_results: List[Dict[str, object]] = []

    for fold_idx, split in enumerate(folds, start=1):
        test_subject = split["test"][0]
        print(f"\n=== Fold {fold_idx}/{len(folds)} | Test: {test_subject} ===")
        print(f"Train subjects: {len(split['train'])}, Val subjects: {len(split['val'])}, Test subjects: 1")

        x_train, y_train = build_split_arrays(
            dataset_root,
            split["train"],
            hp.window_size,
            hp.stride,
            normalization_mode=hp.normalization_mode,
        )
        x_val, y_val = build_split_arrays(
            dataset_root,
            split["val"],
            hp.window_size,
            hp.stride,
            normalization_mode=hp.normalization_mode,
        )
        x_test, y_test = build_split_arrays(
            dataset_root,
            split["test"],
            hp.window_size,
            hp.stride,
            normalization_mode=hp.normalization_mode,
        )

        train_loader = make_loader(x_train, y_train, hp.batch_size, shuffle=True)
        val_loader = make_loader(x_val, y_val, hp.batch_size, shuffle=False)
        test_loader = make_loader(x_test, y_test, hp.batch_size, shuffle=False)

        model = Acc1DCNNTransformer(hp=hp).to(device)
        criterion, loss_cfg = make_criterion(y_train, hp, device)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=hp.learning_rate,
            weight_decay=hp.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=4
        )

        best_val_loss = float("inf")
        best_epoch = -1
        wait = 0
        history_rows: List[Dict[str, float]] = []
        best_state_dict = None
        fold_tag = f"fold_{fold_idx:02d}_{test_subject}"
        best_model_path = save_dir / f"{model_name}_{fold_tag}_best.pt"
        final_model_path = save_dir / f"{model_name}_{fold_tag}.pt"

        for epoch in range(1, hp.epochs + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            scheduler.step(val_metrics["loss"])
            history_rows.append({"epoch": float(epoch), "train_loss": float(train_loss), "val_loss": float(val_metrics["loss"]), "val_accuracy": float(val_metrics["accuracy"]), "val_f1": float(val_metrics["f1"]), "lr": float(optimizer.param_groups[0]["lr"])})

            if val_metrics["loss"] < best_val_loss:
                best_val_loss = val_metrics["loss"]
                best_epoch = epoch
                wait = 0
                best_state_dict = deepcopy(model.state_dict())
                if hp.save_model:
                    torch.save(
                        {
                            "model_state_dict": best_state_dict,
                            "hyperparameters": asdict(hp),
                            "model_name": model_name,
                            "fold_index": fold_idx,
                            "test_subject": test_subject,
                            "best_epoch": best_epoch,
                            "best_val_loss": best_val_loss,
                        },
                        best_model_path,
                    )
            else:
                wait += 1
                if wait >= hp.patience:
                    print(f"Early stopping at epoch {epoch} (best epoch: {best_epoch})")
                    break

        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)

        test_metrics = evaluate(model, test_loader, criterion, device)

        if hp.save_model:
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "hyperparameters": asdict(hp),
                    "model_name": model_name,
                    "fold_index": fold_idx,
                    "test_subject": test_subject,
                    "best_epoch": best_epoch,
                    "best_val_loss": best_val_loss,
                    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
                },
                final_model_path,
            )

        fold_result = {
            "fold_index": fold_idx,
            "test_subject": test_subject,
            "split": split,
            "sample_shapes": {
                "x_train": list(x_train.shape),
                "x_val": list(x_val.shape),
                "x_test": list(x_test.shape),
            },
            "class_distribution": {
                "train": {"non_stress": int(np.sum(y_train == 0)), "stress": int(np.sum(y_train == 1))},
                "val": {"non_stress": int(np.sum(y_val == 0)), "stress": int(np.sum(y_val == 1))},
                "test": {"non_stress": int(np.sum(y_test == 0)), "stress": int(np.sum(y_test == 1))},
            },
            "best_epoch": int(best_epoch),
            "best_val_loss": float(best_val_loss),
            "test_metrics": {k: float(v) for k, v in test_metrics.items()},
            "normalization_mode": hp.normalization_mode,
            "loss_config": loss_cfg,
            "history": history_rows,
            "model_saved": bool(hp.save_model),
            "model_files": {
                "best_model": str(best_model_path) if hp.save_model else None,
                "final_model": str(final_model_path) if hp.save_model else None,
            },
        }
        fold_results.append(fold_result)

        print(
            "Fold test metrics:",
            {
                "accuracy": round(float(test_metrics["accuracy"]), 4),
                "f1": round(float(test_metrics["f1"]), 4),
                "loss": round(float(test_metrics["loss"]), 4),
            },
        )

    summary_df = pd.DataFrame([
        {
            "fold_index": fr["fold_index"],
            "test_subject": fr["test_subject"],
            "accuracy": fr["test_metrics"]["accuracy"],
            "precision": fr["test_metrics"]["precision"],
            "recall": fr["test_metrics"]["recall"],
            "f1": fr["test_metrics"]["f1"],
            "loss": fr["test_metrics"]["loss"],
        }
        for fr in fold_results
    ])

    metric_cols = ["accuracy", "precision", "recall", "f1", "loss"]
    agg_mean = summary_df[metric_cols].mean().to_dict() if len(summary_df) else {}
    agg_std = summary_df[metric_cols].std(ddof=0).to_dict() if len(summary_df) else {}

    metadata = {
        "created_at": datetime.now().isoformat(),
        "model_name": model_name,
        "model_type": "1D CNN + Transformer (PyTorch, LOSO)",
        "dataset_root": str(dataset_root),
        "signal_type": "ACC",
        "sampling_rate_hz": ACC_SAMPLING_RATE_HZ,
        "label_definition": {
            "non_stress": ["Baseline"],
            "stress": ["Logic", "Nback", "Stroop", "Sudoku"],
        },
        "split_rule": "LOSO: 1 test subject + ~10% of remaining (4~5) for validation",
        "total_complete_subjects": len(subjects),
        "num_folds_run": len(fold_results),
        "hyperparameters": asdict(hp),
        "aggregate_metrics": {
            "mean": {k: float(v) for k, v in agg_mean.items()},
            "std": {k: float(v) for k, v in agg_std.items()},
        },
        "fold_results": fold_results,
        "device": str(device),
    }

    metadata_path = save_dir / f"{model_name}_hyperparameters.json"
    with metadata_path.open("w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print("\n=== LOSO Training Complete ===")
    print(f"Saved metadata (hyperparameters + metrics): {metadata_path}")
    if hp.save_model:
        print("Saved per-fold best/final checkpoints under LOSO save_dir")
    if agg_mean:
        print("Aggregate mean metrics:", {k: round(float(v), 4) for k, v in agg_mean.items()})
        print("Aggregate std metrics:", {k: round(float(v), 4) for k, v in agg_std.items()})

    return metadata


# ===== 실행 파트 =====
window_seconds = 60
stride_seconds = 10

hp = HyperParameters(
    random_seed=42,
    val_ratio=0.1,
    min_val_subjects=4,
    max_val_subjects=5,
    window_size=ACC_SAMPLING_RATE_HZ * window_seconds,
    stride=ACC_SAMPLING_RATE_HZ * stride_seconds,
    batch_size=64,
    epochs=50,
    learning_rate=1e-3,
    dropout_rate=0.3,
    weight_decay=1e-4,
    patience=8,
    transformer_heads=8,
    transformer_layers=2,
    transformer_ff_dim=256,
    transformer_dropout=0.1,
    normalization_mode="subject_zscore",
    loss_name="focal",
    focal_gamma=2.0,
    focal_alpha=None,
    save_model=True,
    max_folds=None,  # 빠른 테스트 시 정수로 제한 가능 (예: 3)
)

result_loso = run_loso_training(model_name="acc_1dcnn_transformer_loso_torch", hp=hp)
result_loso

In [ ]:
# LOSO 결과 시각화: Accuracy/F1, Confusion Matrix, Train/Val Loss curves
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "result_loso" not in globals():
    raise RuntimeError("먼저 1번 셀을 실행해 result_loso를 생성해 주세요.")

fold_results = result_loso.get("fold_results", [])
if len(fold_results) == 0:
    raise RuntimeError("fold_results가 비어 있습니다. 1번 셀 학습 결과를 확인해 주세요.")

# 1) Fold metric 요약
metrics_df = pd.DataFrame([
    {
        "fold_index": fr["fold_index"],
        "test_subject": fr["test_subject"],
        "accuracy": fr["test_metrics"]["accuracy"],
        "f1": fr["test_metrics"]["f1"],
        "tn": fr["test_metrics"]["tn"],
        "fp": fr["test_metrics"]["fp"],
        "fn": fr["test_metrics"]["fn"],
        "tp": fr["test_metrics"]["tp"],
    }
    for fr in fold_results
])

acc_mean = float(metrics_df["accuracy"].mean())
f1_mean = float(metrics_df["f1"].mean())
acc_std = float(metrics_df["accuracy"].std(ddof=0))
f1_std = float(metrics_df["f1"].std(ddof=0))

print("=== LOSO Aggregate Metrics ===")
print(f"Accuracy mean±std: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"F1-score mean±std: {f1_mean:.4f} ± {f1_std:.4f}")

plt.figure(figsize=(6, 4))
bars = plt.bar(["Accuracy", "F1-score"], [acc_mean, f1_mean], color=["#4e79a7", "#f28e2b"], edgecolor="black")
plt.ylim(0, 1.0)
plt.title("LOSO Mean Metrics")
plt.ylabel("Score")
plt.grid(axis="y", linestyle="--", alpha=0.4)
for b, v in zip(bars, [acc_mean, f1_mean]):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

# 2) Confusion Matrix (fold 합산 후 row-normalized)
tn = float(metrics_df["tn"].sum())
fp = float(metrics_df["fp"].sum())
fn = float(metrics_df["fn"].sum())
tp = float(metrics_df["tp"].sum())
conf_counts = np.array([[tn, fp], [fn, tp]], dtype=np.float64)
row_sums = conf_counts.sum(axis=1, keepdims=True)
conf_ratio = np.divide(conf_counts, np.maximum(row_sums, 1.0))

print("Confusion Matrix Count [[TN, FP], [FN, TP]]:")
print(conf_counts.astype(int))
print("Confusion Matrix Row Ratio [[TN, FP], [FN, TP]]:")
print(np.round(conf_ratio, 4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(conf_ratio, cmap="Blues", vmin=0.0, vmax=1.0)
ax.set_title("LOSO Confusion Matrix (Row-normalized)")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Non-stress", "Stress"])
ax.set_yticklabels(["Non-stress", "Stress"])
for i in range(conf_ratio.shape[0]):
    for j in range(conf_ratio.shape[1]):
        ax.text(j, i, f"{conf_ratio[i, j]:.3f}", ha="center", va="center", color="black", fontsize=12, fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

# 3) Train/Val Loss 곡선 (fold 평균)
history_frames = []
for fr in fold_results:
    history_rows = fr.get("history", [])
    if len(history_rows) == 0:
        continue

    h = pd.DataFrame(history_rows)
    if "epoch" in h.columns and "train_loss" in h.columns and "val_loss" in h.columns:
        h["fold_index"] = fr["fold_index"]
        history_frames.append(h[["fold_index", "epoch", "train_loss", "val_loss"]])

if len(history_frames) == 0:
    print("history가 없어 Train/Val loss 곡선을 그릴 수 없습니다. 1번 셀을 다시 실행해 주세요.")
else:
    hist_all = pd.concat(history_frames, ignore_index=True)
    hist_mean = hist_all.groupby("epoch", as_index=False)[["train_loss", "val_loss"]].mean()

    plt.figure(figsize=(7, 4))
    plt.plot(hist_mean["epoch"], hist_mean["train_loss"], label="Train Loss", linewidth=2)
    plt.plot(hist_mean["epoch"], hist_mean["val_loss"], label="Val Loss", linewidth=2)
    plt.title("LOSO Mean Train/Val Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(alpha=0.3, linestyle="--")
    plt.legend()
    plt.tight_layout()
    plt.show()

{
    "accuracy_mean": acc_mean,
    "f1_mean": f1_mean,
    "confusion_matrix_count": conf_counts.astype(int).tolist(),
    "confusion_matrix_row_ratio": conf_ratio.tolist(),
}

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# 1. result 변수에 의존하지 않고, 노트북에 있는 find_dataset_root() 함수를 직접 사용합니다.
dataset_root = find_dataset_root()
all_subjects = discover_complete_subjects(dataset_root)

# 2. 전체 데이터에 대해 피험자별 Z-score 정규화 적용하여 로드
print(f"총 {len(all_subjects)}명의 데이터를 로드하여 PCA를 수행합니다... (시간이 조금 걸릴 수 있습니다)")
x_all, y_all = build_split_arrays(
    dataset_root,
    all_subjects,
    window_size=ACC_SAMPLING_RATE_HZ * 60,
    stride=ACC_SAMPLING_RATE_HZ * 10,
    normalization_mode="subject_zscore",
)

# 3. (N, C, L) -> (N, C*L)로 평탄화
x_all_2d = x_all.reshape(x_all.shape[0], -1)

# 4. PCA 모델 생성 및 학습
pca = PCA(n_components=2)
x_pca = pca.fit_transform(x_all_2d)

# 5. 시각화
plt.figure(figsize=(10, 7))
plt.scatter(
    x_pca[y_all == 0, 0],
    x_pca[y_all == 0, 1],
    color="#4e79a7",
    alpha=0.5,
    label="Non-stress (0)",
    edgecolors="none",
    s=20,
)
plt.scatter(
    x_pca[y_all == 1, 0],
    x_pca[y_all == 1, 1],
    color="#f28e2b",
    alpha=0.5,
    label="Stress (1)",
    edgecolors="none",
    s=20,
)

plt.title(
    f"2D PCA of ACC Time-Series (All {len(all_subjects)} Subjects, Subject Z-Score)",
    fontsize=15,
    fontweight="bold",
)
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)", fontsize=12)
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)", fontsize=12)

leg = plt.legend(fontsize=12, markerscale=2)
for lh in leg.legend_handles:
    lh.set_alpha(1)

plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()